# SignBridge — Day 3: ASL CNN Training (Kaggle T4 GPU)

Trains all three ASL models in sequence:
1. Baseline CNN (from scratch)
2. MobileNetV2 Phase 1 (frozen base)
3. MobileNetV2 Phase 2 (fine-tune top 30 layers)

**Before running:**
- Add your GCP service account JSON as a Kaggle Secret named `GCP_SERVICE_ACCOUNT`
  (Kaggle → notebook → Add-ons → Secrets → paste JSON content)
- Enable GPU: Settings → Accelerator → GPU T4 x1
- Enable Internet: Settings → Internet → On

In [ ]:
# ── Cell 1: GPU check ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — check accelerator setting')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q google-cloud-storage==2.10.0 google-auth==2.22.0 scikit-learn seaborn
print('Dependencies installed')

In [ ]:
# ── Cell 3: GCS authentication via Kaggle Secret ─────────────────────────────
import json, os
from kaggle_secrets import UserSecretsClient

secret = UserSecretsClient().get_secret('GCP_SERVICE_ACCOUNT')
sa_path = '/kaggle/working/gcp_sa.json'
with open(sa_path, 'w') as f:
    f.write(secret)
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = sa_path

# Verify
from google.cloud import storage
client = storage.Client()
bucket = client.bucket('signbridge-data')
print('GCS auth OK — bucket:', bucket.name)

In [ ]:
# ── Cell 4: Clone repo and set PYTHONPATH ─────────────────────────────────────
import sys

!git clone https://github.com/Nosher007/signbridge.git /kaggle/working/signbridge 2>&1 | tail -3

REPO = '/kaggle/working/signbridge'
sys.path.insert(0, REPO)
os.environ['PYTHONPATH'] = REPO

print('Repo cloned. Files in src/models:')
!ls /kaggle/working/signbridge/src/models/

In [ ]:
# ── Cell 5: Download ASL images from GCS → local disk ────────────────────────
# ~87k images (~3-4 GB). Takes ~5-8 min on Kaggle.
# This only needs to run ONCE per session.

ASL_LOCAL = '/kaggle/working/asl_train'
os.environ['ASL_TRAIN_DIR'] = ASL_LOCAL

if not os.path.isdir(ASL_LOCAL) or len(os.listdir(ASL_LOCAL)) < 29:
    print('Downloading ASL images from GCS...')
    !gsutil -m cp -r gs://signbridge-data/raw/asl_alphabet/train {ASL_LOCAL}
    # Kaggle puts images one level deeper: asl_train/train/A/ → fix
    inner = os.path.join(ASL_LOCAL, 'train')
    if os.path.isdir(inner):
        import shutil
        for cls in os.listdir(inner):
            shutil.move(os.path.join(inner, cls), ASL_LOCAL)
        os.rmdir(inner)
    print(f'Done. Classes found: {len(os.listdir(ASL_LOCAL))}')
else:
    print(f'Already downloaded. Classes: {len(os.listdir(ASL_LOCAL))}')

!ls {ASL_LOCAL} | head -10

In [ ]:
# ── Cell 6: Verify TF sees the GPU ───────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'TF version: {tf.__version__}')
print(f'GPUs available: {gpus}')
assert len(gpus) > 0, 'No GPU! Enable T4 in Settings → Accelerator'

In [ ]:
# ── Cell 7: Train Baseline CNN ────────────────────────────────────────────────
# ~15-25 min on T4

!cd /kaggle/working/signbridge && python src/models/train_asl.py --model cnn 2>&1

In [ ]:
# ── Cell 8: Train MobileNetV2 Phase 1 (frozen base) ──────────────────────────
# ~10-15 min on T4

!cd /kaggle/working/signbridge && python src/models/train_asl.py --model mobilenetv2 --phase 1 2>&1

In [ ]:
# ── Cell 9: Train MobileNetV2 Phase 2 (fine-tune top 30 layers) ──────────────
# ~15-20 min on T4
# Loads Phase 1 checkpoint from GCS automatically

!cd /kaggle/working/signbridge && python src/models/train_asl.py \
    --model mobilenetv2 \
    --phase 2 \
    --phase1_ckpt gs://signbridge-data/models/asl_mobilenetv2_phase1.keras 2>&1

In [ ]:
# ── Cell 10: Verify all models saved to GCS ──────────────────────────────────
!gsutil ls gs://signbridge-data/models/
!gsutil ls gs://signbridge-data/docs/figures/

In [ ]:
# ── Cell 11: Quick eval summary ──────────────────────────────────────────────
# Run the final MobileNetV2 on 10 held-out test images to confirm predictions

import numpy as np
from google.cloud import storage as gcs
import io

def load_npy(blob_path):
    buf = io.BytesIO(gcs.Client().bucket('signbridge-data').blob(blob_path).download_as_bytes())
    return np.load(buf)

# Load ASL landmark test split (for quick sanity check on MLP)
X_test = load_npy('processed/asl_landmarks/X_test.npy')
y_test = load_npy('processed/asl_landmarks/y_test.npy')
classes = load_npy('processed/asl_landmarks/classes.npy')

# Spot check: show 5 random test samples and their true labels
idx = np.random.choice(len(y_test), 5, replace=False)
print('Sample test labels:', [str(classes[y_test[i]]) for i in idx])
print('\nAll models trained and saved. Day 3 complete.')